# Ders 05 - Ajan Temelli RAG


## Kurulum

Bu not defteri, Microsoft Agent Framework kullanarak Agentic RAG (Retrieval-Augmented Generation) modelini göstermektedir.

**Ön Koşullar:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — Azure AI Search servis uç noktanız
- `AZURE_SEARCH_API_KEY` — Azure AI Search API anahtarınız
- Ortam değişkenleri aracılığıyla yapılandırılmış Azure OpenAI dağıtımı
- Azure CLI ile kimlik doğrulaması yapılmış (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Agentic RAG Nedir?

Geleneksel RAG sabit bir iş akışını takip eder: belgeleri getirir, ardından yanıt üretir. **Agentic RAG** daha ileri giderek ajana bilgiyi **ne zaman** ve **nasıl** alacağı konusunda özerklik verir.

Agentic RAG ile ajan şunları yapabilir:
- Soruya yanıt vermeden önce alım yapılıp yapılmayacağına **karar vermek**
- Hangi veri kaynağına veya araca sorgu yapılacağını **seçmek**
- Getirilen sonuçları **değerlendirmek** ve ilk deneme yetersizse takip eden alımları gerçekleştirmek
- Birden fazla alım adımından gelen bilgileri tutarlı bir yanıt halinde **birleştirmek**

Bu, ajanın statik bir getir-sonra-üret iş akışına kıyasla daha esnek ve doğru olmasını sağlar.


## Bir Arama Aracı Oluşturma

Agentic RAG'de, dış veri kaynakları aracın talep üzerine çağırabileceği **araçlar** olarak sarılır. Bu, araca geri getirmenin zorunlu bir adım olmaktan ziyade, alabileceği başka bir eylem olarak ele alınmasını sağlar.

Aşağıda bir seyahat bilgi tabanı tanımlıyor ve araca destinasyon bilgilerini araştırmak için çağırabileceği bir araç olarak sunuyoruz.


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## RAG Ajanı Oluşturma

Şimdi, **her zaman yanıtlamadan önce bilgi alması** talimatı verilen bir ajan oluşturuyoruz. Ajan, yanıtlarını kendi eğitim verilerine dayanmak yerine bilgi tabanına dayandırmak için `search_travel_knowledge` aracını kullanır.


In [ ]:
agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## Yinelemeli Getirme — Yapıcı-Kontrolcü Deseni

Agentic RAG'ın temel bir avantajı **yinelemeli getirme**dir. Ajan, ilk bulgularını doğrulamak, iyileştirmek veya genişletmek için birden çok arama turu gerçekleştirebilir — "yapıcı-kontrolcü" iş akışına benzer şekilde:

1. **Yapıcı adım**: Ajan ilk bilgileri getirir ve bir yanıt taslağı hazırlar.
2. **Kontrolcü adım**: Ajan, ayrıntıları doğrulamak veya boşlukları doldurmak için ek getirmeler yapar.

Aşağıda, ajan birkaç kez arama yapmaya teşvik eden, birden fazla destinasyonun karşılaştırılmasını gerektiren bir soruyla karşılaşmaktadır.


In [ ]:
checker_agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## Özet

Bu derste Microsoft Agent Framework kullanarak nasıl **Agentic RAG** sistemi kuracağınızı öğrendiniz:

- **Agentic RAG**, ajanların bilgiyi ne zaman alacaklarına otonom şekilde karar vermesini sağlar; böylece bilgi alma sabit değil, dinamik olur.
- **Veri kaynağı olarak araçlar**: Dış bilgi tabanları (Azure AI Search gibi) ajanın çağırabileceği araçlar olarak sarılır.
- **Yinelenen alma**: Maker-checker modeli, ajanın nihai cevabı üretmeden önce birden fazla bilgi alma turu yapmasına — arama, doğrulama ve iyileştirme — olanak tanır.

Üretimde, bellek içi `TRAVEL_KNOWLEDGE_BASE`’i, büyük ölçekli seyahat dokümanı alımı için gerçek bir Azure AI Search dizini ile değiştireceksiniz.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Feragatname**:
Bu belge, AI çeviri hizmeti [Co-op Translator](https://github.com/Azure/co-op-translator) kullanılarak çevrilmiştir. Doğruluk için çaba sarf etsek de, otomatik çevirilerin hatalar veya yanlışlıklar içerebileceğini unutmayınız. Orijinal belge, yazıldığı dildeki haliyle yetkili kaynak olarak kabul edilmelidir. Kritik bilgiler için profesyonel insan çevirisi önerilir. Bu çevirinin kullanımı sonucu ortaya çıkabilecek yanlış anlamalar veya yorum farklılıklarından sorumlu değiliz.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
